# Part 4 — LLM-Powered Feature
## Track C: Model Prediction Explanation Pipeline

**Chosen track: (C) Model Prediction Explanation Pipeline**

This notebook loads the best-performing model saved in Part 3 (`best_model.pkl`, a `SimpleImputer → StandardScaler → RandomForestClassifier` pipeline trained on the cleaned employee dataset from Part 1/2), generates predictions for three hand-crafted employee records, and asks an LLM (via the OpenRouter API) to produce a structured, schema-validated JSON explanation of each prediction for a non-technical HR audience.

**Dataset / model recap:** the classifier predicts whether an employee's `salary` is above the dataset median (`1`) or at/below it (`0`), based on `age`, `years_experience`, `education_level`, `performance_score`, `monthly_hours`, `distance_from_office_km`, `num_projects`, `department`, and `region`.



## 0. Setup

In [ ]:
# Run once if these are not already installed in your environment.
!pip install requests jsonschema pandas scikit-learn joblib tabulate -q

In [ ]:
import os
import re
import json
import requests
import jsonschema
import joblib
import pandas as pd

pd.set_option('display.max_colwidth', None)

## 1. Load the best-performing model (Track C requirement)

Loads `best_model.pkl` produced in Part 3. It is an sklearn `Pipeline` that already contains its own imputer and scaler, so it must be fed **raw, unscaled** feature values in the exact column order it was trained on.

In [ ]:
MODEL_PATH = 'best_model.pkl'

model = joblib.load(MODEL_PATH)
print(model)

# The exact column order the pipeline expects. We read this straight off the fitted
# pipeline instead of hardcoding it, so encode_record() can never drift out of sync
# with what the model was actually trained on.
FEATURE_COLUMNS = list(model.feature_names_in_)
print('\nExpected feature columns (in order):')
print(FEATURE_COLUMNS)

## 2. `encode_record()` — preprocess a raw feature dict into the model's input format

This must reproduce the exact encoding used when the model was trained in Part 2:
- `education_level` is an **ordinal** label encoding: `High School=0 < Bachelors=1 < Masters=2 < PhD=3`. *(Assumption: this is the standard ordering. If your Part 2 README documents a different mapping, update `EDUCATION_ORDER` below to match — the rest of the pipeline does not need to change.)*
- `department` and `region` are **one-hot encoded** with `drop_first=True` (`pd.get_dummies(..., drop_first=True)`, default alphabetical ordering). This was verified directly against `model.feature_names_in_`: the baseline (dropped) categories are `department=Engineering` and `region=Central`.

In [ ]:
EDUCATION_ORDER = {'High School': 0, 'Bachelors': 1, 'Masters': 2, 'PhD': 3}
DEPARTMENT_DUMMIES = ['Finance', 'HR', 'Marketing', 'Sales', 'Support']  # baseline: Engineering
REGION_DUMMIES = ['East', 'North', 'South', 'West']                     # baseline: Central


def encode_record(features: dict) -> pd.DataFrame:
    """Preprocess a raw feature dict into a single-row DataFrame matching the
    column names and order the trained pipeline expects (model.feature_names_in_).

    Parameters
    ----------
    features : dict with keys:
        age, years_experience, education_level (str), performance_score,
        monthly_hours, distance_from_office_km, num_projects,
        department (str), region (str)
    """
    row = {
        'age': features['age'],
        'years_experience': features['years_experience'],
        'education_level': EDUCATION_ORDER[features['education_level']],
        'performance_score': features['performance_score'],
        'monthly_hours': features['monthly_hours'],
        'distance_from_office_km': features['distance_from_office_km'],
        'num_projects': features['num_projects'],
    }
    for dept in DEPARTMENT_DUMMIES:
        row[f'department_{dept}'] = 1 if features['department'] == dept else 0
    for reg in REGION_DUMMIES:
        row[f'region_{reg}'] = 1 if features['region'] == reg else 0

    return pd.DataFrame([row])[FEATURE_COLUMNS]

## 3. Three hand-crafted feature-vector inputs

Chosen to span a range of model confidence: a clear low-probability case, a genuinely borderline case, and a clear high-probability case — this gives the LLM explanation step something interesting to differentiate between.

In [ ]:
sample_inputs = [
    {  # Case 1: young, low experience -> model is confident this is NOT above-median salary
        'age': 27, 'years_experience': 2, 'education_level': 'Bachelors',
        'performance_score': 3.2, 'monthly_hours': 150, 'distance_from_office_km': 20.0,
        'num_projects': 2, 'department': 'Sales', 'region': 'East',
    },
    {  # Case 2: mid-career, borderline -> model is genuinely uncertain
        'age': 49, 'years_experience': 25.0, 'education_level': 'Masters',
        'performance_score': 3.8, 'monthly_hours': 178, 'distance_from_office_km': 11.0,
        'num_projects': 5, 'department': 'Marketing', 'region': 'South',
    },
    {  # Case 3: senior, highly experienced -> model is confident this IS above-median salary
        'age': 52, 'years_experience': 28, 'education_level': 'Masters',
        'performance_score': 4.5, 'monthly_hours': 185, 'distance_from_office_km': 6.0,
        'num_projects': 7, 'department': 'Engineering', 'region': 'Central',
    },
]

predictions = []
for feat in sample_inputs:
    encoded = encode_record(feat)
    pred_class = int(model.predict(encoded)[0])
    pred_proba = float(model.predict_proba(encoded)[0][1])
    predictions.append({'features': feat, 'predicted_class': pred_class, 'probability': round(pred_proba, 4)})
    print(f"Input: {feat}")
    print(f"  -> predicted_class={pred_class}  P(above-median salary)={round(pred_proba, 4)}\n")

## 4. `call_llm()` — reusable OpenRouter API wrapper

The API key is read from the `OPENROUTER_API_KEY` environment variable — it is **never** hardcoded in this notebook. `OPENROUTER_MODEL` defaults to `openrouter/free`, OpenRouter's auto-router that picks a live free model for each request; swap it for a specific model id (e.g. `meta-llama/llama-3.3-70b-instruct:free`) if you want a fixed model for reproducibility.

In [ ]:
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY')
OPENROUTER_URL = 'https://openrouter.ai/api/v1/chat/completions'
OPENROUTER_MODEL = 'openrouter/free'  # or e.g. 'meta-llama/llama-3.3-70b-instruct:free'


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.0, max_tokens: int = 512):
    """Call the OpenRouter chat completions API. Returns the assistant's text content,
    or None if the call fails."""
    if not OPENROUTER_API_KEY:
        print('OPENROUTER_API_KEY environment variable is not set.')
        return None

    payload = {
        'model': OPENROUTER_MODEL,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        'temperature': temperature,
        'max_tokens': max_tokens,
    }
    headers = {
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        'Content-Type': 'application/json',
    }

    response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    if response.status_code != 200:
        print(f'LLM API call failed with status code {response.status_code}: {response.text}')
        return None

    return response.json()['choices'][0]['message']['content']

**Test call** — confirms the connection and key work before building anything on top of it:

In [ ]:
test_output = call_llm(
    system_prompt='You are a helpful assistant.',
    user_prompt='Reply with only the word: hello',
)
print('Test output:', repr(test_output))

## 5. PII guardrail

Runs before every LLM call. Blocks the call outright if the user-facing prompt contains an email address or a phone number.

In [ ]:
def has_pii(text: str) -> bool:
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))


def guarded_call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.0, max_tokens: int = 512):
    """Wraps call_llm() with the PII guardrail. Blocks the call if PII is detected
    in the user prompt."""
    if has_pii(user_prompt):
        print('Input blocked: PII detected.')
        return None
    return call_llm(system_prompt, user_prompt, temperature=temperature, max_tokens=max_tokens)

**Guardrail demonstration** — one input with an email (should be blocked), one clean input (should proceed):

In [ ]:
pii_test_input = (
    'Employee record: age=27, years_experience=2, department=Sales, '
    'contact email jane.doe@example.com'
)
clean_test_input = 'Employee record: age=27, years_experience=2, department=Sales.'

print('--- Test 1: input WITH an email address (expected: blocked) ---')
result_pii = guarded_call_llm('You are a helpful assistant.', pii_test_input)
print('Result:', result_pii)

print('\n--- Test 2: clean input with no PII (expected: proceeds to the LLM call) ---')
result_clean = guarded_call_llm('You are a helpful assistant.', clean_test_input + ' Reply with only the word: ok', max_tokens=10)
print('Result:', result_clean)

## 6. Prompt design

Zero-shot system prompt (Track C uses zero-shot per the assignment spec) instructing the LLM to return only a JSON object with five required scalar fields. `temperature=0` is used for the main explanation pipeline so that, given the same prediction, the explanation is deterministic and reproducible — the model always selects its highest-probability next token rather than sampling, which matters here because we are treating the LLM's output as structured data to be parsed and validated, not as creative text.

In [ ]:
SYSTEM_PROMPT = (
    'You are an assistant that explains machine learning model predictions to a '
    'non-technical HR manager. You will be given: (1) a set of employee feature values, '
    '(2) the model\'s predicted class (0 = salary at or below the company median, '
    '1 = salary above the company median), and (3) the model\'s predicted probability '
    'for class 1. Using only the information provided, explain the prediction in plain '
    'language. Respond with ONLY a single valid JSON object and no other text, markdown, '
    'or commentary. The JSON object must have exactly these five fields: '
    '"prediction_label" (string: a short human-readable label for the prediction, e.g. '
    '"Above-median salary" or "At/below-median salary"), "confidence_level" (string: '
    'one of "low", "medium", or "high", based on how far the predicted probability is '
    'from 0.5), "top_reason" (string: the single feature most likely driving this '
    'prediction, in plain language), "second_reason" (string: the second most likely '
    'driving feature, in plain language), "next_step" (string: one practical '
    'recommendation or action an HR manager could take based on this prediction). '
    'Do not invent facts about the employee beyond the given feature values. Do not '
    'include any text outside the JSON object.'
)

USER_PROMPT_TEMPLATE = (
    'Employee feature values: {features}\n'
    'Model predicted class: {predicted_class} ({class_meaning})\n'
    'Model predicted probability of class 1 (above-median salary): {probability}\n\n'
    'Explain this prediction as instructed.'
)

print(SYSTEM_PROMPT)
print()
print(USER_PROMPT_TEMPLATE)

## 7. Structured output schema + validated explanation pipeline

After each LLM call: strip whitespace, parse with `json.loads()` inside `try/except json.JSONDecodeError`, then validate with `jsonschema.validate()` inside `try/except jsonschema.ValidationError`. On any failure, log the error and return a fallback dict with all five fields set to `None`.

In [ ]:
EXPLANATION_SCHEMA = {
    'type': 'object',
    'properties': {
        'prediction_label': {'type': 'string'},
        'confidence_level': {'type': 'string', 'enum': ['low', 'medium', 'high']},
        'top_reason': {'type': 'string'},
        'second_reason': {'type': 'string'},
        'next_step': {'type': 'string'},
    },
    'required': ['prediction_label', 'confidence_level', 'top_reason', 'second_reason', 'next_step'],
    'additionalProperties': False,
}

FALLBACK_EXPLANATION = {
    'prediction_label': None,
    'confidence_level': None,
    'top_reason': None,
    'second_reason': None,
    'next_step': None,
}


def explain_prediction(features: dict, predicted_class: int, probability: float):
    """Builds the user prompt, runs the PII guardrail, calls the LLM at temperature=0,
    parses + schema-validates the JSON response, and falls back to a null-filled dict
    on any failure. Returns (explanation_dict, status_string, user_prompt, raw_response)."""
    class_meaning = 'above-median salary' if predicted_class == 1 else 'at/below-median salary'
    user_prompt = USER_PROMPT_TEMPLATE.format(
        features=json.dumps(features),
        predicted_class=predicted_class,
        class_meaning=class_meaning,
        probability=round(float(probability), 4),
    )

    if has_pii(user_prompt):
        print('Input blocked: PII detected.')
        return dict(FALLBACK_EXPLANATION), 'blocked (PII detected)', user_prompt, None

    raw_response = call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.0)
    if raw_response is None:
        return dict(FALLBACK_EXPLANATION), 'fail (no response from API)', user_prompt, None

    try:
        parsed = json.loads(raw_response.strip())
    except json.JSONDecodeError as e:
        print('JSON decode error:', e)
        return dict(FALLBACK_EXPLANATION), f'fail (JSONDecodeError: {e})', user_prompt, raw_response

    try:
        jsonschema.validate(instance=parsed, schema=EXPLANATION_SCHEMA)
    except jsonschema.ValidationError as e:
        print('Schema validation error:', e.message)
        return dict(FALLBACK_EXPLANATION), f'fail (ValidationError: {e.message})', user_prompt, raw_response

    return parsed, 'pass', user_prompt, raw_response

## 8. End-to-end demonstration on the three inputs

For each of the three hand-crafted inputs: predict, explain, validate, and print the feature values, predicted class, predicted probability, raw LLM response, and validation outcome.

In [ ]:
demo_rows = []
for feat in sample_inputs:
    encoded = encode_record(feat)
    pred_class = int(model.predict(encoded)[0])
    pred_proba = float(model.predict_proba(encoded)[0][1])

    explanation, status, user_prompt, raw = explain_prediction(feat, pred_class, pred_proba)

    print('=' * 100)
    print('Feature input:', feat)
    print('Predicted class:', pred_class, '| Probability:', round(pred_proba, 4))
    print('Raw LLM response:', raw)
    print('Validation status:', status)
    print('Explanation JSON:', explanation)

    demo_rows.append({
        'Feature Input': feat,
        'Predicted Class': pred_class,
        'Probability': round(pred_proba, 4),
        'Explanation JSON': explanation,
        'Validation Status': status,
    })

Render as a table — copy this straight into the README's 3-row demonstration table:

In [ ]:
demo_df = pd.DataFrame(demo_rows)
print(demo_df.to_markdown(index=False))
demo_df

## 9. Temperature A/B comparison (temperature=0 vs temperature=0.7)

For each of the three inputs, call the LLM twice with the same prompt — once at `temperature=0` and once at `temperature=0.7` — and compare the outputs.

In [ ]:
temp_ab_rows = []
for feat in sample_inputs:
    encoded = encode_record(feat)
    pred_class = int(model.predict(encoded)[0])
    pred_proba = float(model.predict_proba(encoded)[0][1])
    class_meaning = 'above-median salary' if pred_class == 1 else 'at/below-median salary'
    user_prompt = USER_PROMPT_TEMPLATE.format(
        features=json.dumps(feat),
        predicted_class=pred_class,
        class_meaning=class_meaning,
        probability=round(pred_proba, 4),
    )

    out_t0 = guarded_call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.0)
    out_t07 = guarded_call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.7)

    print('Input:', feat)
    print('  temp=0.0  ->', out_t0)
    print('  temp=0.7  ->', out_t07)
    print()

    temp_ab_rows.append({
        'Input': feat,
        'Output at temp=0': out_t0,
        'Output at temp=0.7': out_t07,
        'Key difference': '',  # fill in manually in the README after reading both outputs
    })

temp_ab_df = pd.DataFrame(temp_ab_rows)
print(temp_ab_df.to_markdown(index=False))
temp_ab_df

**Why temperature=0 is more deterministic:** at `temperature=0` the model always selects the single highest-probability next token at each step, so the same prompt produces the same (or near-identical) output every time. **Why temperature=0.7 introduces variability:** it samples from a wider slice of the model's probability distribution over next tokens, so lower-probability (but still plausible) tokens can be selected, producing different wording, different emphasis, or occasionally a different `confidence_level` or `top_reason` across runs — which is exactly why temperature=0 was chosen for the structured-output pipeline in Section 7/8, and temperature=0.7 is only used here for comparison.